# kagglehub — introductory notebook

M3 requires a minimal notebook per selected tool. This one covers **kagglehub**, the
library the project uses to fetch public Kaggle Datasets without an account or API token.

It shows three things: what a download returns, that repeated downloads hit the cache, and
why the project checks a SHA-256 against the registry instead of trusting the download.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

## Download a public dataset

`dataset_download("<owner>/<slug>/versions/<version>")` returns the local directory
for a pinned dataset version. The example uses a small CSV from the project registry.

In [2]:
import kagglehub

from swsn.registry import load_registry, sha256_file

registry = load_registry("../data/datasets.yaml")
dataset = registry.datasets["heart_failure"]
handle = f"{dataset.kaggle_slug}/versions/{dataset.version}"
path = kagglehub.dataset_download(handle)
print("dataset:", handle)

dataset: fedesoriano/heart-failure-prediction/versions/1


## Files and cache behaviour

The returned path lives inside kagglehub's cache. A repeat request for the same pinned
version can reuse the cached files. The example checks that both calls resolve to the
same directory.

In [3]:
from pathlib import Path

second = kagglehub.dataset_download(handle)
print("same directory:", Path(path) == Path(second))
print("files        :", sorted(p.name for p in Path(path).iterdir()))

same directory: True
files        : ['heart.csv']


## Verify a checksum

`data/datasets.yaml` records a SHA-256 per file. Verify the cached file against the
registry before using it so the experiment runs on the intended data.

In [4]:
actual = sha256_file(Path(path) / "heart.csv")
expected = dataset.sha256["heart.csv"]
print("registry :", expected)
print("actual   :", actual)
if actual != expected:
    raise ValueError("SHA-256 mismatch for heart.csv")
print("checksum verified")

registry : 948420b084d8a3a0ca42b8419fce9aee175879e43f8aedf712377899a67aa49b
actual   : 948420b084d8a3a0ca42b8419fce9aee175879e43f8aedf712377899a67aa49b
checksum verified


## Relation to the project's download step

`make data` calls `swsn.download`, which uses kagglehub and then copies and verifies the
files into `data/raw/<dataset>/` so that every loader reads one layout. `make data
OFFLINE=1` skips the network and only verifies what is already on disk. kagglehub's cache
is an implementation detail; the registry and the checksum are the contract.

## Reproducing

```bash
make setup
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/tool_kagglehub.ipynb
```